In [1]:
# Autoreload custom modules
%load_ext autoreload
%autoreload 2

In [2]:
import sys, os

import pandas as pd
import json

import numpy as np
import jax
import jax.numpy as jnp
import optax

import time
import importlib
import matplotlib.pyplot as plt

In [3]:
# import custom modules
sys.path.append('..')

from models.models import DecoderOnlyTransformer
from util.data_loader import load_data, encode, decode, get_batch
from util.optimization import train_step, loss_and_metrics, create_train_state

In [4]:
# load dataset
train_data, test_data, ctoi, itoc, vocab_size = load_data(encoded_path='../data/encoded.pkl')

In [5]:
# create a basic training loop just for sanity check that the model runs
niters = 100
batch_size = 16
seq_len = 128
learning_rate = 1e-3
key = jax.random.PRNGKey(0)

model, params = create_train_state(key, vocab_size, 256, 4, 4, seq_len)
tx = optax.adamw(learning_rate)
opt_state = tx.init(params)


for i in range(niters):
    input, target = get_batch(train_data, batch_size, seq_len)

    params_new, opt_state_new, metrics = train_step(model, params, opt_state, input, target, tx)

    # update params and opt_state
    params = params_new
    opt_state = opt_state_new

    # Evaluate on test set every 10 iters
    if i % 10 == 0:
        B_test, T_test = 1024, 64
        test_input, test_target = get_batch(test_data, batch_size, seq_len)
        test_logits = model.apply({'params': params}, test_input, deterministic=True) # Prevent dropout during evaluation
        test_loss, test_metrics = loss_and_metrics(test_logits, test_target)
        print(f"Iter {i}, Train Loss: {metrics['loss']:.4f}, Test Loss: {test_loss:.4f}")


Iter 0, Train Loss: 4.0511, Test Loss: 5.3508


KeyboardInterrupt: 